Gather some information about data. What type of features the data have, which features are important, what is the hidden structure of the data.

In [0]:
# from csv to spark dataframe
df = spark.read.format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("/Volumes/health_insurance/default/health_insurance/Health Insurance Cross Sell Prediction/train.csv")

In [0]:
# convert spark dataframe to pandas dataframe
pdf = df.limit(10000).toPandas() # limit to 10000 rows for faster processing

# overview of data
print(pdf.info())
print(pdf.describe())

In [0]:
# target variable
import matplotlib.pyplot as plt
import seaborn as sns 

sns.countplot(data=pdf, x = "Response")
plt.title("Response Distribution (0 = No, 1 = Yes)")
display(plt.show())

In [0]:
# categorical feature vs target
fig, axes = plt.subplots(1,3, figsize=(15,4))
sns.barplot(data=pdf, x = "Vehicle_Damage", y = "Response", ax=axes[0])
axes[0].set_title("Vehicle Damage vs Response")
sns.barplot(data=pdf, x = "Previously_Insured", y ="Response", ax = axes[1])
axes[1].set_title("Previously Insured vs Response")
sns.barplot(data=pdf, x = "Gender", y = "Response", ax = axes[2])
axes[2].set_title("Gender vs Response")
display(plt.show())
# there is no clear correlation between categorical features and target
# numerical feature distribution
pdf[['Age', 'Annual_Premium', 'Vintage']].hist(figsize=(12, 8))
display(plt.show())



**After the first look we can see:**
- Data is imbalanced, only about 16% customer are curious 
- Vehicle_damage = Yes those customers are also curious
- If they are previously insured then they are not interested
- Age , Premium is also responsible to take decision

**Data preprocessing and feature Engineering:**
- Now transfer **categorical data** to Numerical data, then scale Numerical Data and finally split the data by training data and testing data.

In [0]:
# Categorical feature encoding

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

# Categorical Column
categorical_cols = ['Gender', 'Vehicle_Age', 'Vehicle_Damage']


In [0]:
# Create Pipeline
indexers = [StringIndexer(inputCol=col, outputCol=col+"_index", handleInvalid="keep") for col in categorical_cols]
encoders = [OneHotEncoder(inputCol=col+"_index", outputCol=col+"_ohe") for col in categorical_cols]

# numeric columns
numeric_cols = ['Age', 'Annual_Premium', 'Vintage']

# All features vector
assembler_inputs = [col+"_ohe" for col in categorical_cols] + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

# Scaling (important for logistic regression)
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")

# Pipeline
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])

# transform
transformed_df = pipeline.fit(df).transform(df)
display(transformed_df.select("id", "Gender", "Gender_index", "Gender_ohe", "scaled_features"))

In [0]:
# Now split the data into training and test sets
# Spark DF -> pandas
import pandas as pd
final_pdf = transformed_df.select('scaled_features', 'Response').toPandas()

# split features and target
X = pd.DataFrame(final_pdf['scaled_features'].tolist())
y = final_pdf['Response']

# split into training and test sets (fix imbalance problem by using stratify)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

**Now we train our baseline models:**
- There are three different models (Logistic Regression, Random Forest, XGBoost) we will run and find difference between their results
- As our data is imbalanced so we should focus on Recall and F1-Score.

In [0]:
%pip install xgboost

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

def evaluate_model(y_test, y_pred, y_prob= None):
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
    if y_prob is not None:
        print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")
    print(confusion_matrix(y_test, y_pred))

In [0]:
# Model-1: Logistic Regression
lr = LogisticRegression(class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print("-- Logistic Regression --")
evaluate_model(y_test, y_pred_lr, lr.predict_proba(X_test)[:,1])

In [0]:
# Model-2: Random Forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("-- Random Forest --")
evaluate_model(y_test, y_pred_rf, rf.predict_proba(X_test)[:,1])

In [0]:
# Model-3: XGBoost
# scale_pos_weight for imbalance handle
scale = len(y_train[y_train==0]) / len(y_train[y_train==1])
xgb_model = xgb.XGBClassifier(scale_pos_weight=scale, random_state=42, eval_metric = 'logloss')
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
print("-- XGBoost --")
evaluate_model(y_test, y_pred_xgb, xgb_model.predict_proba(X_test)[:,1])

**Hyperparameter tuning by using Optuna:**
- Find out all hyperparameter of XGBoost automatically, which helps to increase performance

In [0]:
%pip install optuna


In [0]:
import optuna
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step = 50),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.001, 10.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.001, 10.0, log=True),
        'scale_pos_weight': scale,
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # F1-score maximization
    return f1_score(y_test, y_pred)

# Create Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("Best parameters:", study.best_params)
print("Best F1-score:", study.best_value)

# Train the best model by using best parameters
best_xgb = xgb.XGBClassifier(**study.best_params,  random_state = 42)
best_xgb.fit(X_train, y_train)



**Track the experiment using MLflow.**
- keep record of every model, parameters, matrix and everything.

In [0]:
import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name="XGBoost_Optuna_Best") as run:

    # parameter log
    mlflow.log_params(study.best_params)

    # matrix log
    y_pred = best_xgb.predict(X_test)
    y_prob = best_xgb.predict_proba(X_test)[:,1]
    mlflow.log_metric("f1_score", f1_score(y_test, y_pred))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_prob))
    mlflow.log_metric("precision", precision_score(y_test, y_pred))
    mlflow.log_metric("recall", recall_score(y_test, y_pred))

    # model log with signature and input_example (required for Unity Catalog)
    from mlflow.models import infer_signature
    signature = infer_signature(X_train, best_xgb.predict(X_train))
    mlflow.sklearn.log_model(
        best_xgb,
        "xgboost_model",
        signature=signature,
        input_example=X_train[:5]
    )

    # Feature importance log 
    importance = pd.DataFrame({
        'feature': [f'feature_{i}' for i in range(X_train.shape[1])],
        'importance': best_xgb.feature_importances_
    })
    importance.to_csv("/tmp/feature_importance.csv")
    mlflow.log_artifact("/tmp/feature_importance.csv")

# To register model in Unity Catalog
model_uri = f"runs:/{run.info.run_id}/xgboost_model"
mlflow.register_model(model_uri, "health_insurance.default.insurance_model")



**Explainable AI:**
- What is the reason behind every decision of the model, we can briefly describe this by using **SHAP** and **LIME**.

**SHAP:**
- SHAP measures of contributions each feature has in the model. It can be applied for multiple forms data prediction.

**LIME:**
- LIME is short for Local Interpretable Model-Agnostic Explanations. It is able to explain any model without needing to 'peak' into it.

In [0]:
%pip install shap

**Plot_label**  **    ** **Original_name**
-    0  ->  Male
-    1  ->  Female
-    2  ->  vehicle damage(<1 year)
-    3  ->  vehicle damage(1-2 year)
-    4  ->  vehicle damage(>2 year)
-    5  ->  vehicle damage(yes)
-    6  ->  vehicle damage(no)
-    7  ->  Age
-    8  ->  Annual_Premium
-    9  ->  Vintage

In [0]:
import shap

# Workaround for SHAP categorical feature incompatibility:
# Create a copy of the model without categorical features enabled
import copy
shap_model = copy.deepcopy(best_xgb)
shap_model.enable_categorical = False

# create shap explainer
explainer = shap.TreeExplainer(shap_model, X_train, feature_perturbation="interventional")
shap_values = explainer(X_test[:100])       # took 100 samples for performance

# Set feature names for plotting
shap_values.feature_names = [str(col) for col in X_train.columns]

# Which features are important for global interpretablity
shap.plots.bar(shap_values, max_display=10)
plt.savefig("/tmp/shap_bar.png")
mlflow.log_artifact("/tmp/shap_bar.png")

# How to affect beeswarm features on prediction
shap.plots.beeswarm(shap_values, max_display=10)
plt.savefig("/tmp/shap_beeswarm.png")
mlflow.log_artifact("/tmp/shap_beeswarm.png")

# Description for local interpretablity
shap.plots.waterfall(shap_values[0])   # shape_vales[0] = first sample
plt.savefig("/tmp/shap_waterfall.png")
mlflow.log_artifact("/tmp/shap_waterfall.png")

shap.plots.waterfall(shap_values[5])   # shape_vales[5] = sixth sample
plt.savefig("/tmp/shap_waterfall_5.png")
mlflow.log_artifact("/tmp/shap_waterfall_5.png")

** What we got from SHAP explanation:**
- Using SHAP, I identified the top factors driving customer interest. Feature 5 was the most strongest positive driver for a purchase, while Feature 1 and 4 had a negative impact. I also validated the model's decision for individual customers by tracing exactly how each feature contributed to their final score.

**LIME Implementation**

In [0]:
%pip install lime

In [0]:
import lime
from lime.lime_tabular import LimeTabularExplainer

# create Lime Explainer
lime_explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=[f'feature_{i}' for i in range(X_train.shape[1])],
    class_names = ['Not Interested', 'Interested'],
    mode='classification',
    verbose= True
)

# describe for targeted customer
for idx in [0, 5, 10]:
    exp = lime_explainer.explain_instance(
        X_test.values[idx],
        best_xgb.predict_proba,
        num_features = 10
    )
    exp.show_in_notebook(show_table=True)
    plt.savefig(f"/tmp/lime_explanation{idx}.png")
    mlflow.log_artifact(f"/tmp/lime_explanation{idx}.png")

**SHAP vs LIME**

| Aspect | SHAP | LIME |
| :--- | :--- | :--- |
| Consistency | Consistent (same output for same input) | Can vary across runs |
| Speed | Slower (especially for large models) | Faster |
| Theory | Game Theory (Shapley values) | Local surrogate model |
| Use Case | Global + Local explanations | Primarily local explanations |